In [4]:
import os
import numpy as np
import cv2
import nibabel as nib
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import Sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

2026-03-22 13:26:44.493944: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774186004.696471     553 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774186004.751764     553 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774186005.222453     553 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774186005.222492     553 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774186005.222495     553 computation_placer.cc:177] computation placer alr

In [5]:
!pip install nibabel -q

In [12]:


IMG_SIZE = 224

# --- Healthy images ---
healthy_path = "/kaggle/input/datasets/nandeeshhu/pancrease-ct-segmenatation/images"

# --- Tumor images ---
tumor_path = "/kaggle/input/datasets/rksrank1/pancreatic-cancer/Task07_Pancreas/imagesTr"

# List tumor NIfTI files
tumor_files = [f for f in os.listdir(tumor_path) if f.endswith(".nii.gz") and not f.startswith("._")]
print("Number of tumor files:", len(tumor_files))

Number of tumor files: 0


In [14]:
class CTDataGenerator(Sequence):
    def __init__(self, healthy_folders, tumor_files, batch_size=32, img_size=IMG_SIZE):
        self.healthy_folders = healthy_folders
        self.tumor_files = tumor_files
        self.batch_size = batch_size
        self.img_size = img_size
        self.samples = self._prepare_samples()

    def _prepare_samples(self):
        samples = []

        # Healthy images
        for cls in ['negative', 'positive']:
            cls_folder = os.path.join(healthy_path, cls)
            for f in os.listdir(cls_folder):
                if f.endswith(".png") or f.endswith(".jpg"):
                    samples.append((os.path.join(cls_folder, f), 0))  # 0 = healthy

        # Tumor slices
        for f in self.tumor_files:
            nii = nib.load(os.path.join(tumor_path, f))
            volume = nii.get_fdata()
            patient_id = f.replace(".nii.gz","")
            for i in range(volume.shape[2]):
                samples.append(((patient_id, i), 1))  # 1 = tumor

        np.random.shuffle(samples)
        return samples

    def __len__(self):
        return int(np.ceil(len(self.samples)/self.batch_size))

    def __getitem__(self, idx):
        batch_samples = self.samples[idx*self.batch_size:(idx+1)*self.batch_size]
        X_batch = []
        y_batch = []

        for s, label in batch_samples:
            if label == 0:
                # Healthy image
                img = cv2.imread(s)
                img = cv2.resize(img, (self.img_size, self.img_size))
            else:
                # Tumor slice
                patient_id, slice_idx = s
                nii = nib.load(os.path.join(tumor_path, f"{patient_id}.nii.gz"))
                volume = nii.get_fdata()
                slice_img = volume[:,:,slice_idx]
                slice_img = (slice_img - np.min(slice_img)) / (np.max(slice_img) + 1e-8)
                slice_img = (slice_img * 255).astype(np.uint8)
                slice_img = cv2.resize(slice_img, (self.img_size, self.img_size))
                img = cv2.cvtColor(slice_img, cv2.COLOR_GRAY2RGB)

            X_batch.append(img)
            y_batch.append(label)

        X_batch = np.array(X_batch)/255.0
        y_batch = np.array(y_batch)
        return X_batch, y_batch

In [16]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model.output
x = GlobalAveragePooling2D()(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [17]:
from sklearn.model_selection import train_test_split

train_gen = CTDataGenerator(healthy_folders=['negative','positive'], tumor_files=tumor_files, batch_size=16)
# If you have a separate test split, create test_gen

model.fit(train_gen, epochs=10)

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10


I0000 00:00:1774186374.860344     607 service.cc:152] XLA service 0x7cb06c002ce0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1774186374.860383     607 service.cc:160]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1774186378.724536     607 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-03-22 13:33:07.204511: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-22 13:33:07.401234: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1774186398.493418     607 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the proc

 982/1184 ━━━━━━━━━━━━━━━━━━━━ 36s 182ms/step - accuracy: 0.9957 - loss: 0.0083

2026-03-22 13:36:30.252496: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-03-22 13:36:30.448612: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


1184/1184 ━━━━━━━━━━━━━━━━━━━━ 288s 202ms/step - accuracy: 0.9964 - loss: 0.0070
Epoch 2/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 85s 72ms/step - accuracy: 1.0000 - loss: 6.0811e-07
Epoch 3/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 87s 74ms/step - accuracy: 1.0000 - loss: 2.2255e-07
Epoch 4/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 84s 71ms/step - accuracy: 1.0000 - loss: 1.0966e-07
Epoch 5/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 90s 76ms/step - accuracy: 1.0000 - loss: 5.7103e-08
Epoch 6/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 89s 75ms/step - accuracy: 1.0000 - loss: 3.0810e-08
Epoch 7/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 97s 82ms/step - accuracy: 1.0000 - loss: 1.6723e-08
Epoch 8/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 109s 92ms/step - accuracy: 1.0000 - loss: 9.8361e-09
Epoch 9/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 85s 72ms/step - accuracy: 1.0000 - loss: 5.8028e-09
Epoch 10/10
1184/1184 ━━━━━━━━━━━━━━━━━━━━ 90s 76ms/step - accuracy: 1.0000 - loss: 3.4594e-09


In [23]:
def predict_patient(model, patient_file):
    slices = load_tumor_patient(patient_file)
    X = np.array(slices)/255.0
    preds = model.predict(X, verbose=0)
    return preds.mean()  # average across slices

In [26]:
y_true, y_pred = [], []

for X_batch, y_batch in test_gen:
    preds = model.predict(X_batch)
    preds_label = (preds > 0.5).astype(int)
    y_true.extend(y_batch)
    y_pred.extend(preds_label)

y_true = np.array(y_true)
y_pred = np.array(y_pred)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1-score:", f1_score(y_true, y_pred))
print("ROC-AUC:", roc_auc_score(y_true, y_pred))

NameError: name 'test_gen' is not defined

In [18]:
model.save("/kaggle/working/pancreas_mobilenet.h5")